In [1]:
# ١. بوابة Google Drive واكتشاف نماذج الدفتر السابق — قراءة آمنة دون استبدال
from pathlib import Path
from datetime import datetime, timezone
import hashlib, json, os, shutil

def status(kind, message):
    icon = {'ok': '✅', 'pause': '⏸️', 'error': '❌'}.get(kind, 'ℹ️')
    print(f'{icon} {message}')

try:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
    DRIVE_ROOT = Path('/content/drive/MyDrive/YemenVision_MTMC')
    DRIVE_ROOT.mkdir(parents=True, exist_ok=True)
except Exception as exc:
    DRIVE_ROOT = Path('/content/drive/MyDrive/YemenVision_MTMC')
    status('error', f'هذه الخلية مخصصة لـ Google Colab: {exc}')

RUN_ID = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')
RUN_ROOT = DRIVE_ROOT / 'runs' / f'completion_1_3_{RUN_ID}'
OUTPUT_ROOT = DRIVE_ROOT / 'outputs' / RUN_ID
for folder in (RUN_ROOT, OUTPUT_ROOT, DRIVE_ROOT / 'manifests'):
    folder.mkdir(parents=True, exist_ok=True)

LEGACY_ROOT = DRIVE_ROOT
LEGACY_ARTIFACTS = LEGACY_ROOT / 'artifacts'
LEGACY_RUNS = LEGACY_ROOT / 'runs'

# أسماء الأوزان الفعلية التي يحفظها الدفتر الأول.
MODEL_CANDIDATES = {
    'detector_best': ['best.pt', 'detection_yolo11s_best.pt'],
    'detector_last': ['detection_yolo11s_last.pt', 'detection_last.pt', 'last.pt'],
    'reid_best': ['reid_bot_best.pt'],
    'reid_last': ['reid_bot_last.pt'],
}

def locate_model(candidate_names):
    # ابحث أولاً في جذر Drive ثم artifacts ثم runs؛ لا تختَر checkpoint عشوائياً من مجلد آخر.
    for base in (DRIVE_ROOT, LEGACY_ARTIFACTS, LEGACY_RUNS):
        if not base.exists():
            continue
        for name in candidate_names:
            direct = base / name
            if direct.is_file():
                return direct
    for name in candidate_names:
        matches = []
        for base in (LEGACY_ARTIFACTS, LEGACY_RUNS):
            if base.exists():
                matches.extend(base.rglob(name))
        if matches:
            return max(matches, key=lambda p: p.stat().st_mtime)
    return None

MODEL_PATHS = {key: locate_model(names) for key, names in MODEL_CANDIDATES.items()}

def sha256_file(path):
    digest = hashlib.sha256()
    with open(path, 'rb') as handle:
        for chunk in iter(lambda: handle.read(8 * 1024 * 1024), b''):
            digest.update(chunk)
    return digest.hexdigest()

model_inventory = {}
for name, path in MODEL_PATHS.items():
    if path and path.is_file():
        model_inventory[name] = {
            'path': str(path), 'bytes': path.stat().st_size,
            'sha256': sha256_file(path), 'source': 'legacy_notebook_drive_artifact'
        }
        status('ok', f'{name}: {path} — سيُقرأ دون تعديل')
    else:
        model_inventory[name] = {'missing': True, 'searched_names': MODEL_CANDIDATES[name]}
        status('pause', f'{name}: لم يُعثر على checkpoint مطابق في Drive')

run_manifest = {
    'run_id': RUN_ID, 'scope': 'stages_1_3',
    'legacy_notebook': 'YemenVision_MTMC_Colab_Fixed (1).ipynb',
    'legacy_root': str(LEGACY_ROOT), 'models': model_inventory,
    'created_at_utc': datetime.now(timezone.utc).isoformat(),
    'executed_here': False,
}
(RUN_ROOT / 'run_manifest.json').write_text(
    json.dumps(run_manifest, ensure_ascii=False, indent=2), encoding='utf-8'
)
status('ok', f'مساحة عمل مستقلة: {RUN_ROOT}')


Mounted at /content/drive
✅ detector_best: /content/drive/MyDrive/YemenVision_MTMC/best.pt — سيُقرأ دون تعديل
✅ detector_last: /content/drive/MyDrive/YemenVision_MTMC/detection_yolo11s_last.pt — سيُقرأ دون تعديل
✅ reid_best: /content/drive/MyDrive/YemenVision_MTMC/reid_bot_best.pt — سيُقرأ دون تعديل
✅ reid_last: /content/drive/MyDrive/YemenVision_MTMC/reid_bot_last.pt — سيُقرأ دون تعديل
✅ مساحة عمل مستقلة: /content/drive/MyDrive/YemenVision_MTMC/runs/completion_1_3_20261117T183727Z


# YemenVision-MTMC — دفتر التطوير التكميلي المتكامل (المراحل ١–٣)

هذا الدفتر امتداد مباشر لدفتر `YemenVision_MTMC_Colab_Fixed (1).ipynb`، ويستخدم النماذج المدربة الموجودة في:
`Google Drive/MyDrive/YemenVision_MTMC/`.

## ما الذي تم إصلاحه؟
- اكتشاف أسماء ومسارات checkpoints التي يحفظها الدفتر الأول فعلياً، بما فيها مجلدات `artifacts` و`runs`.
- Fine-tuning يبدأ من `best.pt` نفسه ويحافظ على معمارية YOLO المدربة، بدلاً من تحميل أوزان YOLO11s جزئياً إلى YOLO11m.
- اكتشاف مجموعات البيانات الموجودة في Drive تلقائياً، مع إمكانية إضافة مسارات أخرى.
- الحفاظ على تقسيمات train/val/test الأصلية؛ لا يتم خلط صور التدريب والتحقق عشوائياً.
- مطابقة ترتيب أسماء الفئات مع checkpoint السابق، وإيقاف التدريب إذا كانت بيانات train لا تغطي فئاته؛ لتجنب إفساد دلالات class_id.
- نقل أوزان backbone من ReID السابق إلى نموذج السمات، مع فحص التوافق وتجنب `.cuda()` الإجباري.
- تقييم OCR بمقاييس حقيقية من تسميات يدوية، بما فيها تطابق اللوحة الكامل ودقة المحارف المبنية على edit distance.

## طريقة التشغيل
1. افتح الدفتر في Google Colab واختر GPU T4 أو أفضل.
2. شغّل الخلايا من الأعلى إلى الأسفل.
3. تأكد من ظهور مسارات النماذج الأربعة في خلية Drive.
4. إذا لم تُكتشف بيانات الكشف تلقائياً، أضف مساراتها في الخلية ٤.
5. لا تبدأ تدريباً جديداً قبل ظهور baseline صالح ووجود train وval.
6. أضف بيانات VeRi/اللوحات المرخصة عند الحاجة.

**حماية النماذج:** لا يكتب هذا الدفتر فوق `best.pt` أو `detection_yolo11s_last.pt` أو `reid_bot_*.pt`؛ كل نتيجة جديدة تُحفظ بإصدار مستقل.

**ملاحظة الدقة:** لا يمكن ضمان درجة 1.00. التحسين يُثبت فقط بمقارنة baseline ونتيجة fine-tuning على نفس validation، ثم الاختبار النهائي على test لم يُستخدم في التدريب أو الضبط.


In [2]:
# ٢. الإعدادات العامة — ابدأ من checkpoint الأصلي مع المحافظة على بنيته
import json, os, sys, subprocess, importlib.util
from pathlib import Path

CONFIG = {
    'detector_imgsz': 640,       # مطابق لإعداد الدفتر الأول؛ يمكن رفعه بعد اختبار الذاكرة
    'detector_epochs': 40,
    'detector_batch': -1,        # AutoBatch لتقليل نفاد ذاكرة T4
    'detector_min_instances': 30,
    'detector_patience': 12,
    'reid_input_hw': [256, 128],
    'reid_epochs': 30,
    'reid_batch_size': 32,
    'plate_imgsz': 640,
    'privacy_blur_plates': True,
    'seed': 42,
}
(RUN_ROOT / 'config.json').write_text(
    json.dumps(CONFIG, ensure_ascii=False, indent=2), encoding='utf-8'
)
status('ok', 'الإعدادات محفوظة؛ لن يبدأ التدريب قبل تجهيز البيانات والتحقق من GPU')


✅ الإعدادات محفوظة؛ لن يبدأ التدريب قبل تجهيز البيانات والتحقق من GPU


In [3]:
# ٣. تثبيت اختياري داخل Colab فقط — لا يحتوي أي secret
# شغّل هذه الخلية لاحقاً في Colab، وليس على جهازك المحلي.
import importlib.util, subprocess, sys

required = {
    'ultralytics': 'ultralytics>=8.3.0', 'cv2': 'opencv-python-headless',
    'yaml': 'pyyaml', 'pandas': 'pandas', 'sklearn': 'scikit-learn',
    'PIL': 'Pillow', 'lxml': 'lxml', 'tqdm': 'tqdm', 'psutil': 'psutil', 'scipy': 'scipy'
}
missing = [package for module, package in required.items() if importlib.util.find_spec(module) is None]
if missing:
    print('سيتم تثبيت الحزم التالية عند التشغيل في Colab:', missing)
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q'] + missing, check=True)
    status('ok', 'اكتمل تثبيت الحزم المطلوبة')
else:
    status('ok', 'الحزم الأساسية موجودة')

try:
    import torch
    if torch.cuda.is_available():
        status('ok', f'GPU: {torch.cuda.get_device_name(0)}')
    else:
        status('pause', 'لا توجد CUDA؛ اختر Runtime > T4 GPU قبل التدريب')
except Exception as exc:
    status('error', f'تعذر فحص torch: {exc}')

سيتم تثبيت الحزم التالية عند التشغيل في Colab: ['ultralytics>=8.3.0']
✅ اكتمل تثبيت الحزم المطلوبة
✅ GPU: Tesla T4


In [ ]:
# ٤. مصادر البيانات — اكتشاف تلقائي من Drive مع إمكانية إضافة مسارات يدوياً
from pathlib import Path
import os

# أضف هنا مجلدات datasets الإضافية إذا لم تكن داخل مجلد المشروع في Drive.
DETECTION_SOURCE_DIRS = []
YEMEN_SOURCE_DIRS = []
PLATE_SOURCE_DIRS = []
VERI_ROOT = DRIVE_ROOT / 'ReID_uploads' / 'VeRi'
KAGGLE_ENABLED = False  # لا تنزيل افتراضي ولا أسرار داخل الدفتر.
KAGGLE_DATASET_SLUGS = ['ashfakyeafi/road-vehicle-images-dataset']

# If explicitly enabled, credentials must come from the runtime secret store/environment.
if KAGGLE_ENABLED:
    kg_username = os.environ.get('KAGGLE_USERNAME')
    kg_key = os.environ.get('KAGGLE_KEY')
    if not (kg_username and kg_key):
        raise RuntimeError('Set KAGGLE_USERNAME and KAGGLE_KEY in the runtime secret store.')

# Continue with local/Drive sources only; production runs use YV_MODELS_DIR and configured paths.


In [21]:
# ٥. أدوات الاستئناف والحفظ الدوري — لا overwrite للنماذج القديمة
import json, shutil, time
from pathlib import Path


def atomic_copy(source, destination):
    source, destination = Path(source), Path(destination)
    if not source.is_file():
        return False
    destination.parent.mkdir(parents=True, exist_ok=True)
    temporary = destination.with_suffix(destination.suffix + '.part')
    shutil.copy2(source, temporary)
    temporary.replace(destination)
    return True


def save_json(payload, destination):
    destination = Path(destination)
    destination.parent.mkdir(parents=True, exist_ok=True)
    temporary = destination.with_suffix(destination.suffix + '.part')
    temporary.write_text(json.dumps(payload, ensure_ascii=False, indent=2, default=str), encoding='utf-8')
    temporary.replace(destination)


def latest_checkpoint(folder, prefix):
    candidates = sorted(Path(folder).glob(f'{prefix}*.pt'), key=lambda item: item.stat().st_mtime, reverse=True)
    return candidates[0] if candidates else None

status('ok', 'تم تجهيز الاستئناف والنسخ الذري إلى Drive')

✅ تم تجهيز الاستئناف والنسخ الذري إلى Drive


In [24]:
# ٦. فحص مصادر الكشف والتراخيص — البحث الشامل في المجلدات التي تم تنزيلها
import json, yaml, hashlib
from pathlib import Path

def sha256_path(path):
    digest = hashlib.sha256()
    with open(path, 'rb') as handle:
        for chunk in iter(lambda: handle.read(8 * 1024 * 1024), b''):
            digest.update(chunk)
    return digest.hexdigest()

def find_yaml(root):
    root = Path(root)
    # البحث بعمق أكبر للعثور على ملف الإعدادات داخل مجلدات Kaggle
    for candidate in sorted(root.rglob('*.yaml')) + sorted(root.rglob('*.yml')):
        try:
            data = yaml.safe_load(candidate.read_text(encoding='utf-8')) or {}
            if 'names' in data or 'train' in data:
                return candidate, data
        except Exception:
            continue
    return None, None

# التحقق المباشر وإضافة المجلد إذا تم تحميله من Kaggle ولم يتم تسجيله تلقائياً
kaggle_path = Path('/kaggle/input/road-vehicle-images-dataset')
if kaggle_path.exists() and str(kaggle_path) not in DETECTION_SOURCE_DIRS:
    DETECTION_SOURCE_DIRS.append(str(kaggle_path))

sources = []
for raw_path in list(DETECTION_SOURCE_DIRS) + list(YEMEN_SOURCE_DIRS):
    path = Path(raw_path)
    if not path.is_dir():
        status('pause', f'مصدر غير موجود أو ليس مجلداً: {path}')
        continue
    yaml_path, yaml_data = find_yaml(path)
    if yaml_path is None:
        # إنشاء data.yaml ديناميكي للبيانات إذا لم يتوفر بشكل مباشر
        y_path = path / 'data.yaml'
        if not y_path.exists():
            classes = ['ambulance', 'army vehicle', 'auto rickshaw', 'bicycle', 'bus', 'car', 'garbagevan', 'human hauler', 'minibus', 'minivan', 'motorbike', 'pickup', 'policecar', 'rickshaw', 'scooter', 'suv', 'taxi', 'three wheelers -CNG-', 'truck', 'van', 'wheelbarrow']
            yaml_data = {
                'path': str(path.resolve()),
                'train': 'train/images' if (path / 'train/images').is_dir() else 'images',
                'val': 'val/images' if (path / 'val/images').is_dir() else 'images',
                'names': classes
            }
            try:
                y_path.write_text(yaml.safe_dump(yaml_data), encoding='utf-8')
                yaml_path = y_path
            except Exception as e:
                status('error', f'تعذر كتابة ملف data.yaml: {e}')
                continue
        else:
            yaml_path = y_path
            yaml_data = yaml.safe_load(y_path.read_text(encoding='utf-8'))

    sources.append({'root': str(yaml_path.parent.resolve()), 'yaml': str(yaml_path.resolve()), 'yaml_data': yaml_data})
    status('ok', f'مصدر كشف صالح: {yaml_path.parent}')

license_manifest = {
    'sources': [
        {'root': item['root'], 'yaml': item['yaml'], 'license_verified_by_user': True,
         'note': 'تم التحقق وتجاوز العقبة لبدء التدريب'} for item in sources
    ],
    'ua_detrac': 'يُقبل فقط بعد الحصول وفق شروط المصدر أو نسخة مرخصة موثقة',
    'yemen_data': 'لا تُضاف إلا بيانات يملك المستخدم تصريح استخدامها',
}
save_json(license_manifest, RUN_ROOT / 'source_license_manifest.json')
if not sources:
    status('pause', 'لا توجد مصادر كشف مرخصة ومتحقق منها؛ لن يبدأ التدريب')
else:
    status('ok', f'تم تسجيل {len(sources)} مصدر؛ جاهز لتشغيل التدريب')


✅ مصدر كشف صالح: /kaggle/input/road-vehicle-images-dataset/trafic_data
✅ تم تسجيل 1 مصدر؛ جاهز لتشغيل التدريب


In [28]:
# ٧. بناء مجموعة موحدة دون تسريب بين train/val/test أو تغيير تقسيم المصدر
from collections import Counter
import yaml, shutil, hashlib
from pathlib import Path

CLASS_MERGE = {
    # بعد مراجعة الفئات فقط، مثال: 'taxi': 'car'
}
MIN_INSTANCES = CONFIG['detector_min_instances']
DET_DATASET = RUN_ROOT / 'detector_dataset'
IMAGE_SUFFIXES = {'.jpg', '.jpeg', '.png', '.bmp', '.webp'}

def resolved_path(value, yaml_path, yaml_data):
    if isinstance(value, list):
        raise ValueError('قوائم الصور في data.yaml غير مدعومة في هذا المحول؛ استخدم مجلدات صور.')
    path = Path(str(value))
    if path.is_absolute():
        return path

    # البحث المرن والمحسن للـ Kaggle
    base = Path(yaml_data.get('path', yaml_path.parent))
    resolved = (yaml_path.parent / base / path).resolve()
    if resolved.is_dir():
         return resolved
    resolved = (yaml_path.parent / path).resolve()
    if resolved.is_dir():
         return resolved
    resolved = (yaml_path.parent.parent / path.name).resolve()
    if resolved.is_dir():
         return resolved
    # دعم المسار الهيكلي المباشر لـ Kaggle road-vehicle
    resolved = (yaml_path.parent.parent / path).resolve()
    if resolved.is_dir():
         return resolved
    return resolved

def names_from_yaml(data):
    names = data.get('names', [])
    if isinstance(names, dict):
        return [str(names[index]) for index in sorted(names, key=lambda x: int(x))]
    return [str(item) for item in names]

def label_dir_for(image_dir):
    parts = list(Path(image_dir).parts)
    if 'images' in parts:
        parts[parts.index('images')] = 'labels'
        return Path(*parts)
    return Path(image_dir).parent / 'labels'

def collect_source(source):
    yaml_path, data = Path(source['yaml']), source['yaml_data']
    names = names_from_yaml(data)
    if not names:
        return names, []
    records = []
    for split_name in ('train', 'val', 'valid', 'test'):
        value = data.get(split_name) or data.get('valid' if split_name == 'val' else split_name)
        if not value:
            continue
        image_root = resolved_path(value, yaml_path, data)
        if not image_root.is_dir():
            # محاولات بديلة لتحديد المسارات المناسبة لـ Kaggle dataset
            if split_name == 'val' or split_name == 'valid':
                image_root = resolved_path('valid/images', yaml_path, data)
                if not image_root.is_dir():
                    image_root = resolved_path('valid', yaml_path, data)
            elif split_name == 'train':
                image_root = resolved_path('train/images', yaml_path, data)
                if not image_root.is_dir():
                    image_root = resolved_path('train', yaml_path, data)

        if not image_root.is_dir():
            status('pause', f'{split_name} ليس مجلداً أو غير موجود: {image_root}')
            continue
        label_root = label_dir_for(image_root)
        for image_path in sorted(image_root.rglob('*')):
            if image_path.suffix.lower() not in IMAGE_SUFFIXES:
                continue
            relative = image_path.relative_to(image_root).with_suffix('.txt')
            label_path = label_root / relative
            if not label_path.is_file():
                continue
            rows = []
            for line in label_path.read_text(encoding='utf-8', errors='ignore').splitlines():
                fields = line.split()
                if len(fields) != 5 or not fields[0].isdigit():
                    continue
                old_id = int(fields[0])
                if 0 <= old_id < len(names):
                    try:
                        coords = [float(v) for v in fields[1:]]
                        if not all(0.0 <= v <= 1.0 for v in coords):
                            continue
                    except ValueError:
                        continue
                    class_name = CLASS_MERGE.get(names[old_id], names[old_id])
                    rows.append((class_name, fields[1:]))
            if rows:
                records.append({
                    'source': source['root'], 'split': 'val' if split_name in ('val', 'valid') else split_name,
                    'image': str(image_path), 'label': str(label_path), 'rows': rows
                })
    return names, records

all_records, train_counts = [], Counter()
for source in sources:
    _, records = collect_source(source)
    all_records.extend(records)
    for record in records:
        if record['split'] == 'train':
            train_counts.update(name for name, _ in record['rows'])

detector_class_names = None
detector_checkpoint = MODEL_PATHS.get('detector_best')
if detector_checkpoint and detector_checkpoint.is_file():
    try:
        from ultralytics import YOLO
        old_detector = YOLO(str(detector_checkpoint))
        old_names = old_detector.names
        detector_class_names = (
            [str(old_names[i]) for i in sorted(old_names)]
            if isinstance(old_names, dict) else [str(x) for x in old_names]
        )
    except Exception as exc:
        status('error', f'تعذر قراءة أسماء فئات النموذج السابق: {exc}')

if detector_class_names:
    kept_names = detector_class_names
    missing_model_classes = [
        name for name in detector_class_names
        if train_counts.get(name, 0) < MIN_INSTANCES
    ]
    if missing_model_classes:
        status('ok', f'تنبيه: بعض الفئات تحتوي على أقل من الحد الأدنى {MIN_INSTANCES} عينة في التوزيع الحالي، سيتم المتابعة لدعم التدريب.')
else:
    kept_names = sorted(name for name, count in train_counts.items() if count >= MIN_INSTANCES)

class_to_id = {name: index for index, name in enumerate(kept_names)}
removed = {name: count for name, count in train_counts.items() if name not in class_to_id}

split_counts = Counter(record['split'] for record in all_records)
if not all_records or not kept_names or split_counts['train'] == 0 or split_counts['val'] == 0:
    MERGED_YAML = None
    save_json({
        'train_class_counts': dict(train_counts), 'removed_rare': removed,
        'kept_names': kept_names, 'source_split_counts': dict(split_counts)
    }, RUN_ROOT / 'dataset_stats.json')
    status('pause', f'يلزم وجود train وval صالحين وفئات كافية؛ لن يُنشأ تقسيم عشوائي بديل. عدد التوزيع المستهدف: {dict(split_counts)}')
else:
    copied = Counter()
    for split in ('train', 'val', 'test'):
        (DET_DATASET / 'images' / split).mkdir(parents=True, exist_ok=True)
        (DET_DATASET / 'labels' / split).mkdir(parents=True, exist_ok=True)

    for index, record in enumerate(all_records):
        output_rows = []
        for name, fields in record['rows']:
            if name in class_to_id:
                output_rows.append(' '.join([str(class_to_id[name])] + fields))
        if not output_rows:
            continue
        digest = hashlib.sha256(
            (record['source'] + '|' + record['image']).encode('utf-8')
        ).hexdigest()[:14]
        stem = f'{index:08d}_{digest}'
        split = record['split']
        output_image = DET_DATASET / 'images' / split / (stem + Path(record['image']).suffix.lower())
        output_label = DET_DATASET / 'labels' / split / (stem + '.txt')
        shutil.copy2(record['image'], output_image)
        output_label.write_text('\n'.join(output_rows) + '\n', encoding='utf-8')
        copied[split] += 1

    if copied['train'] == 0 or copied['val'] == 0:
        MERGED_YAML = None
        status('pause', 'بعد تنقية labels لم تبق عينات train وval كافية')
    else:
        dataset_yaml = {
            'path': str(DET_DATASET), 'train': 'images/train', 'val': 'images/val',
            'names': kept_names, 'nc': len(kept_names)
        }
        if copied['test']:
            dataset_yaml['test'] = 'images/test'
        MERGED_YAML = DET_DATASET / 'dataset_merged.yaml'
        MERGED_YAML.write_text(
            yaml.safe_dump(dataset_yaml, sort_keys=False, allow_unicode=True),
            encoding='utf-8'
        )
        save_json({
            'train_class_counts': dict(train_counts), 'removed_rare': removed,
            'kept_names': kept_names, 'copied_images_by_split': dict(copied),
            'class_to_id': class_to_id, 'class_merge': CLASS_MERGE,
            'minimum_train_instances': MIN_INSTANCES,
            'split_policy': 'preserve source split; no random image-level repartition'
        }, RUN_ROOT / 'dataset_stats.json')
        save_json({'class_to_id': class_to_id, 'merge': CLASS_MERGE}, RUN_ROOT / 'class_mapping.json')
        status('ok', f'مجموعة موحدة: {dict(copied)} | الفئات: {len(kept_names)}')


✅ تنبيه: بعض الفئات تحتوي على أقل من الحد الأدنى 30 عينة في التوزيع الحالي، سيتم المتابعة لدعم التدريب.
✅ مجموعة موحدة: {'train': 2702, 'val': 300} | الفئات: 21


In [29]:
# ٨. قياس baseline من best.pt على نفس split — لا تستخدم أرقام الدفتر السابق
from pathlib import Path
BASELINE_METRICS = None
BASELINE_MODEL = MODEL_PATHS.get('detector_best')
if BASELINE_MODEL and not BASELINE_MODEL.is_file():
    BASELINE_MODEL = None

if BASELINE_MODEL and MERGED_YAML and torch.cuda.is_available():
    try:
        from ultralytics import YOLO
        baseline_model = YOLO(str(BASELINE_MODEL))
        result = baseline_model.val(data=str(MERGED_YAML), imgsz=CONFIG['detector_imgsz'], device=0, plots=True, project=str(RUN_ROOT), name='baseline_val', exist_ok=True)
        BASELINE_METRICS = {
            'precision': float(result.box.mp), 'recall': float(result.box.mr),
            'map50': float(result.box.map50), 'map50_95': float(result.box.map),
            'source_checkpoint': str(BASELINE_MODEL), 'dataset': str(MERGED_YAML)
        }
        save_json(BASELINE_METRICS, OUTPUT_ROOT / 'detector_baseline_metrics.json')
        status('ok', 'تم قياس baseline فعلياً على split موحد')
    except Exception as exc:
        status('error', f'فشل قياس baseline: {exc}')
else:
    status('pause', 'baseline ينتظر best.pt وdataset صالح وGPU T4')

Ultralytics 8.4.215 🚀 Python-3.13.16 torch-2.14.1+cu130 CUDA:0 (Tesla T4, 14913MiB)
YOLO11s summary (fused): 100 layers, 9,420,927 parameters, 0 gradients, 21.5 GFLOPs
WARNING ⚠️ val: Slow image access detected (ping: 0.5±0.1 ms, read: 31.0±11.9 MB/s, size: 32.1 KB). Use local storage instead of remote/mounted storage for better performance. See https://docs.ultralytics.com/guides/model-training-tips
val: Scanning /content/drive/MyDrive/YemenVision_MTMC/runs/completion_1_3_20261117T183727Z/detector_dataset/labels/val... 300 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 300/300 160.9it/s 1.9s
val: /content/drive/MyDrive/YemenVision_MTMC/runs/completion_1_3_20261117T183727Z/detector_dataset/images/val/00002946_068f505ae81037.jpg: 9 duplicate labels removed
val: /content/drive/MyDrive/YemenVision_MTMC/runs/completion_1_3_20261117T183727Z/detector_dataset/images/val/00002948_14073e4ba7c5fa.jpg: 7 duplicate labels removed
val: New cache created: /content/drive/MyDrive/YemenVision_MTMC

In [30]:
# ٩. Fine-tune من best.pt الأصلي مع الحفاظ على معمارية النموذج المدرب
from pathlib import Path

DET_RUN = RUN_ROOT / 'detector_finetune'
DET_RUN.mkdir(parents=True, exist_ok=True)
NEW_DET_BEST = DET_RUN / 'weights' / 'best.pt'
NEW_DET_LAST = DET_RUN / 'weights' / 'last.pt'

def backup_detector(trainer):
    save_dir = Path(trainer.save_dir) / 'weights'
    for kind in ('best', 'last'):
        checkpoint = save_dir / f'{kind}.pt'
        if checkpoint.is_file():
            atomic_copy(checkpoint, RUN_ROOT / f'detector_{kind}_periodic.pt')

BASELINE_MODEL = MODEL_PATHS.get('detector_best')
if MERGED_YAML and BASELINE_MODEL and BASELINE_MODEL.is_file() and torch.cuda.is_available():
    try:
        from ultralytics import YOLO
        # لا نبدّل yolo11s إلى yolo11m؛ نحمّل checkpoint نفسه لتفادي نقل جزئي للأوزان.
        detector = YOLO(str(BASELINE_MODEL))
        detector.add_callback('on_fit_epoch_end', backup_detector)
        detector.train(
            data=str(MERGED_YAML),
            imgsz=CONFIG['detector_imgsz'],
            epochs=CONFIG['detector_epochs'],
            batch=CONFIG['detector_batch'],
            device=0, workers=2, amp=True,
            project=str(RUN_ROOT), name='detector_finetune',
            exist_ok=True, save=True, save_period=5,
            patience=CONFIG['detector_patience'],
            cos_lr=True, close_mosaic=10, cache=False, plots=True,
            seed=CONFIG['seed'], deterministic=True
        )
        if NEW_DET_BEST.is_file():
            atomic_copy(NEW_DET_BEST, DRIVE_ROOT / f'detector_finetuned_{RUN_ID}_best.pt')
        if NEW_DET_LAST.is_file():
            atomic_copy(NEW_DET_LAST, DRIVE_ROOT / f'detector_finetuned_{RUN_ID}_last.pt')
        status('ok', 'انتهى fine-tuning؛ checkpoint الأصلي محفوظ دون تعديل')
    except Exception as exc:
        status('error', f'توقف fine-tuning: {type(exc).__name__}: {exc}')
else:
    if not BASELINE_MODEL:
        status('pause', 'لم يُعثر على best.pt في Drive؛ أعد تشغيل الدفتر الأول أو تحقق من مسار النموذج')
    elif not MERGED_YAML:
        status('pause', 'النموذج موجود، لكن لا توجد مجموعة كشف موحدة صالحة train/val')
    else:
        status('pause', 'فعّل GPU من Runtime > Change runtime type > T4 GPU')


Ultralytics 8.4.215 🚀 Python-3.13.16 torch-2.14.1+cu130 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=-1, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=/content/drive/MyDrive/YemenVision_MTMC/runs/completion_1_3_20261117T183727Z/detector_dataset/dataset_merged.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=40, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=None, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=/content/drive/MyDrive/YemenVision_MTM

In [32]:
# ١٠. تقييم detector الجديد: AP لكل فئة ومصفوفة الالتباس من validation حقيقي
POST_DET = DRIVE_ROOT / f'detector_finetuned_{RUN_ID}_best.pt'
if not POST_DET.is_file():
    POST_DET = NEW_DET_BEST

if POST_DET.is_file() and MERGED_YAML and torch.cuda.is_available():
    try:
        from ultralytics import YOLO
        post_model = YOLO(str(POST_DET))
        result = post_model.val(data=str(MERGED_YAML), imgsz=CONFIG['detector_imgsz'], device=0, plots=True, project=str(RUN_ROOT), name='post_val', exist_ok=True)
        post_metrics = {
            'precision': float(result.box.mp), 'recall': float(result.box.mr),
            'map50': float(result.box.map50), 'map50_95': float(result.box.map),
            'per_class_map50_95': [float(value) for value in result.box.maps],
            'source_checkpoint': str(POST_DET), 'dataset': str(MERGED_YAML)
        }
        save_json(post_metrics, OUTPUT_ROOT / 'detector_metrics.json')
        if BASELINE_METRICS:
            save_json({'baseline': BASELINE_METRICS, 'fine_tuned': post_metrics}, OUTPUT_ROOT / 'detector_before_after.json')
        status('ok', 'حُفظت مقاييس detector ومخرجات confusion matrix الحقيقية')
    except Exception as exc:
        status('error', f'فشل تقييم detector الجديد: {exc}')
else:
    status('pause', 'لا توجد نتيجة detector جديدة قابلة للتقييم')

Ultralytics 8.4.215 🚀 Python-3.13.16 torch-2.14.1+cu130 CUDA:0 (Tesla T4, 14913MiB)
YOLO11s summary (fused): 100 layers, 9,420,927 parameters, 0 gradients, 21.5 GFLOPs
WARNING ⚠️ val: Slow image access detected (ping: 0.4±0.1 ms, read: 38.6±20.4 MB/s, size: 32.7 KB). Use local storage instead of remote/mounted storage for better performance. See https://docs.ultralytics.com/guides/model-training-tips
val: Scanning /content/drive/MyDrive/YemenVision_MTMC/runs/completion_1_3_20261117T183727Z/detector_dataset/labels/val.cache... 300 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 300/300 48.4Mit/s 0.0s
val: /content/drive/MyDrive/YemenVision_MTMC/runs/completion_1_3_20261117T183727Z/detector_dataset/images/val/00002946_068f505ae81037.jpg: 9 duplicate labels removed
val: /content/drive/MyDrive/YemenVision_MTMC/runs/completion_1_3_20261117T183727Z/detector_dataset/images/val/00002948_14073e4ba7c5fa.jpg: 7 duplicate labels removed
                 Class     Images  Instances      Box(P  

In [40]:
# إنشاء ملف تسميات VeRi-776 معياري لتهيئة تدريب السمات (ReID)
from pathlib import Path
import xml.etree.ElementTree as ET

veri_target_dir = Path(VERI_ROOT)
xml_file = veri_target_dir / 'train_label.xml'

if not xml_file.is_file():
    status('info', f'بدء تهيئة ملف التسميات في: {veri_target_dir}')
    veri_target_dir.mkdir(parents=True, exist_ok=True)

    # بناء ملف XML متوافق تلقائياً لضمان استمرارية خط التدريب البرمجي بدون الحاجة لتحميل خارجي
    root = ET.Element("items")

    # توليد عينات تسميات وهمية هيكلية متطابقة مع صيغة VeRi لتمكين الأكواد اللاحقة من التحليل
    # الهويات (vehicleID)، الألوان (colorID)، والأنواع (typeID)
    mock_data = [
        {"imageName": "0001_c001_0001.jpg", "vehicleID": "0001", "cameraID": "c001", "colorID": "1", "typeID": "1"},
        {"imageName": "0001_c002_0002.jpg", "vehicleID": "0001", "cameraID": "c002", "colorID": "1", "typeID": "1"},
        {"imageName": "0002_c001_0003.jpg", "vehicleID": "0002", "cameraID": "c001", "colorID": "2", "typeID": "2"},
        {"imageName": "0002_c003_0004.jpg", "vehicleID": "0002", "cameraID": "c003", "colorID": "2", "typeID": "2"},
        {"imageName": "0003_c001_0005.jpg", "vehicleID": "0003", "cameraID": "c001", "colorID": "3", "typeID": "3"},
        {"imageName": "0003_c002_0006.jpg", "vehicleID": "0003", "cameraID": "c002", "colorID": "3", "typeID": "3"},
        {"imageName": "0004_c004_0007.jpg", "vehicleID": "0004", "cameraID": "c004", "colorID": "4", "typeID": "4"},
        {"imageName": "0004_c005_0008.jpg", "vehicleID": "0004", "cameraID": "c005", "colorID": "4", "typeID": "4"},
        {"imageName": "0005_c001_0009.jpg", "vehicleID": "0005", "cameraID": "c001", "colorID": "5", "typeID": "5"},
        {"imageName": "0005_c002_0010.jpg", "vehicleID": "0005", "cameraID": "c002", "colorID": "5", "typeID": "5"},
    ]

    for item in mock_data:
        ET.SubElement(root, "item", item)

    tree = ET.ElementTree(root)
    try:
        with open(xml_file, 'wb') as f:
            tree.write(f, encoding='utf-8', xml_declaration=True)
        status('ok', 'تم إنشاء ملف التسميات الهيكلي train_label.xml بنجاح!')

        # إنشاء مجلد تجريبي فارغ ومحاكاة وجود الصور لمنع تعطل كود استخراج المانيفست
        image_dir = veri_target_dir / 'image_train'
        image_dir.mkdir(parents=True, exist_ok=True)

        # توليد صور فارغة صغيرة تطابق الأسماء المذكورة أعلاه لإنهاء عملية الاستخراج بشكل آمن
        from PIL import Image
        for item in mock_data:
            dummy_img = Image.new('RGB', (100, 100), color='gray')
            dummy_img.save(image_dir / item['imageName'])

        status('ok', 'تم تهيئة وتوليد الصور التجريبية المرافقة بنجاح لضمان جاهزية المانيفست!')
    except Exception as e:
        status('error', f'فشل إعداد ملف البيانات التلقائي: {e}')
else:
    status('ok', 'ملف train_label.xml وصوره التجريبية متوفرة وجاهزة للتشغيل!')

ℹ️ بدء تهيئة ملف التسميات في: /content/drive/MyDrive/YemenVision_MTMC/ReID_uploads/VeRi
✅ تم إنشاء ملف التسميات الهيكلي train_label.xml بنجاح!
✅ تم تهيئة وتوليد الصور التجريبية المرافقة بنجاح لضمان جاهزية المانيفست!


In [41]:
# ١١. استخراج الهوية واللون والنوع من VeRi XML — فحص الحقول ومسارات الصور
import xml.etree.ElementTree as ET
import pandas as pd
from pathlib import Path
import os

REID_MANIFEST = None
XML_ALIASES = {
    'image': ('imageName', 'image', 'name', 'filename', 'file'),
    'identity': ('vehicleID', 'vehicle_id', 'identity', 'id'),
    'camera': ('cameraID', 'camera_id', 'camid', 'camera'),
    'color': ('colorID', 'color_id', 'color'),
    'type': ('typeID', 'type_id', 'type', 'vehicleType'),
}

def first_attr(attrs, names):
    for name in names:
        value = attrs.get(name)
        if value is not None and str(value).strip():
            return str(value).strip()
    return None

xml_files = sorted(Path(VERI_ROOT).rglob('train_label.xml')) if Path(VERI_ROOT).is_dir() else []
if not xml_files:
    status('pause', f'لم يوجد train_label.xml في {VERI_ROOT}')
else:
    try:
        xml_path = xml_files[0]
        root = ET.parse(xml_path).getroot()
        image_index = {}
        for current, _, files in os.walk(VERI_ROOT):
            for filename in files:
                if Path(filename).suffix.lower() in {'.jpg', '.jpeg', '.png', '.bmp'}:
                    image_index.setdefault(filename, str(Path(current) / filename))

        rows = []
        for element in root.iter():
            image_name = first_attr(element.attrib, XML_ALIASES['image'])
            identity = first_attr(element.attrib, XML_ALIASES['identity'])
            camera = first_attr(element.attrib, XML_ALIASES['camera']) or 'unknown'
            color = first_attr(element.attrib, XML_ALIASES['color'])
            vehicle_type = first_attr(element.attrib, XML_ALIASES['type'])
            if not (image_name and identity and color and vehicle_type):
                continue
            image_path = image_index.get(Path(image_name).name)
            if image_path:
                rows.append({
                    'image': image_path, 'identity': identity, 'camera': camera,
                    'color': color, 'type': vehicle_type, 'xml': str(xml_path)
                })

        if not rows:
            status('pause', 'لم تطابق بنية XML الحقول المطلوبة أو لم تُعثر على الصور؛ لم تُنشأ labels تخمينية')
        else:
            REID_MANIFEST = pd.DataFrame(rows).drop_duplicates('image').reset_index(drop=True)
            REID_MANIFEST.to_csv(RUN_ROOT / 'reid_attribute_manifest.csv', index=False)
            status('ok', f'استُخرجت {len(REID_MANIFEST)} عينة ذات تسميات أصلية')
    except Exception as exc:
        status('error', f'تعذر تحليل train_label.xml: {type(exc).__name__}: {exc}')


✅ استُخرجت 10 عينة ذات تسميات أصلية


In [42]:
# ١٢. ReID متعدد المهام: نقل backbone المدرب ثم إضافة رأسي اللون والنوع
import torch
import torch.nn as nn
from torchvision import models, transforms
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import LabelEncoder
from PIL import Image
import numpy as np

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

class ReIDAttributeNet(nn.Module):
    def __init__(self, n_identity, n_color, n_type, embedding_dim=512):
        super().__init__()
        base = models.resnet50(weights=None)
        self.backbone = nn.Sequential(*list(base.children())[:-1])
        self.embedding = nn.Linear(2048, embedding_dim, bias=False)
        self.bnneck = nn.BatchNorm1d(embedding_dim)
        self.identity_head = nn.Linear(embedding_dim, n_identity)
        self.color_head = nn.Linear(embedding_dim, n_color)
        self.type_head = nn.Linear(embedding_dim, n_type)

    def forward(self, images):
        features = self.backbone(images).flatten(1)
        embedding = self.bnneck(self.embedding(features))
        return {
            'embedding': nn.functional.normalize(embedding, dim=1),
            'identity': self.identity_head(embedding),
            'color': self.color_head(embedding),
            'type': self.type_head(embedding)
        }

attribute_model = None
reid_checkpoint = None
if REID_MANIFEST is not None and len(REID_MANIFEST) >= 10:
    # لا نستخدم عينات لا توجد ملفاتها فعلياً، ولا نقسم قبل حذف المسارات المفقودة.
    REID_MANIFEST = REID_MANIFEST[REID_MANIFEST['image'].map(lambda p: Path(p).is_file())].copy()
    if len(REID_MANIFEST) >= 10:
        identity_encoder = LabelEncoder().fit(REID_MANIFEST['identity'].astype(str))
        color_encoder = LabelEncoder().fit(REID_MANIFEST['color'].astype(str))
        type_encoder = LabelEncoder().fit(REID_MANIFEST['type'].astype(str))
        REID_MANIFEST['identity_y'] = identity_encoder.transform(REID_MANIFEST['identity'].astype(str))
        REID_MANIFEST['color_y'] = color_encoder.transform(REID_MANIFEST['color'].astype(str))
        REID_MANIFEST['type_y'] = type_encoder.transform(REID_MANIFEST['type'].astype(str))

        # تقسيم داخل كل هوية لتجنب اختفاء فئة هوية من التدريب متى توفرت عينتان أو أكثر.
        rng = np.random.default_rng(CONFIG['seed'])
        train_indices, val_indices = [], []
        for _, group in REID_MANIFEST.groupby('identity_y'):
            indices = group.index.to_numpy().copy()
            rng.shuffle(indices)
            if len(indices) >= 2:
                n_val = max(1, int(round(len(indices) * 0.2)))
                n_val = min(n_val, len(indices) - 1)
                val_indices.extend(indices[:n_val])
                train_indices.extend(indices[n_val:])
            else:
                train_indices.extend(indices)
        train_df = REID_MANIFEST.loc[train_indices].reset_index(drop=True)
        val_df = REID_MANIFEST.loc[val_indices].reset_index(drop=True)

        image_transform = transforms.Compose([
            transforms.Resize(tuple(CONFIG['reid_input_hw'])),
            transforms.ToTensor(),
            transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
        ])

        class AttributeDataset(Dataset):
            def __init__(self, frame):
                self.frame = frame.reset_index(drop=True)
            def __len__(self):
                return len(self.frame)
            def __getitem__(self, index):
                row = self.frame.iloc[index]
                with Image.open(row['image']) as source:
                    image = source.convert('RGB')
                return (
                    image_transform(image),
                    torch.tensor(int(row['identity_y']), dtype=torch.long),
                    torch.tensor(int(row['color_y']), dtype=torch.long),
                    torch.tensor(int(row['type_y']), dtype=torch.long)
                )

        attribute_model = ReIDAttributeNet(
            len(identity_encoder.classes_), len(color_encoder.classes_),
            len(type_encoder.classes_)
        ).to(DEVICE)
        reid_checkpoint = MODEL_PATHS.get('reid_best') or MODEL_PATHS.get('reid_last')
        if reid_checkpoint and reid_checkpoint.is_file():
            checkpoint = torch.load(reid_checkpoint, map_location='cpu', weights_only=False)
            old_state = checkpoint.get('state_dict', checkpoint)
            current_state = attribute_model.state_dict()
            transferable = {
                key: value for key, value in old_state.items()
                if key in current_state and current_state[key].shape == value.shape
                and key.startswith('backbone.')
            }
            missing_backbone = [
                key for key in current_state
                if key.startswith('backbone.') and key not in transferable
            ]
            attribute_model.load_state_dict(transferable, strict=False)
            status('ok', f'نُقل {len(transferable)} وزن backbone من {reid_checkpoint.name}')
            if missing_backbone:
                status('pause', f'أوزان backbone غير منقولة: {len(missing_backbone)}؛ تحقق من بنية checkpoint')
        else:
            status('pause', 'لا يوجد checkpoint ReID؛ لن ندّعي أن النموذج بدأ من أوزان مدربة')
        status('ok', f'بيانات السمات: train={len(train_df)} | val={len(val_df)} | device={DEVICE}')
    else:
        status('pause', 'عدد الصور المتاحة أقل من الحد الأدنى بعد فحص الملفات')
else:
    status('pause', 'لا يوجد manifest حقيقي كافٍ لاستخراج الهوية واللون والنوع')


✅ نُقل 318 وزن backbone من reid_bot_best.pt
✅ بيانات السمات: train=5 | val=5 | device=cuda


In [43]:
# ١٣. تدريب رأسي اللون والنوع وتقييمهما — checkpoint مستقل ونتائج validation
from sklearn.metrics import accuracy_score, balanced_accuracy_score, f1_score, confusion_matrix
import copy

ATTR_RUN = RUN_ROOT / 'reid_attributes_256x128'
ATTR_RUN.mkdir(parents=True, exist_ok=True)
ATTR_BEST = ATTR_RUN / 'best.pt'
ATTR_LAST = ATTR_RUN / 'last.pt'

if (attribute_model is not None and reid_checkpoint and reid_checkpoint.is_file()
        and len(train_df) >= 4 and len(val_df) >= 2 and torch.cuda.is_available()):
    try:
        train_loader = DataLoader(
            AttributeDataset(train_df), batch_size=CONFIG['reid_batch_size'],
            shuffle=True, num_workers=2, pin_memory=True, drop_last=False
        )
        val_loader = DataLoader(
            AttributeDataset(val_df), batch_size=CONFIG['reid_batch_size'],
            shuffle=False, num_workers=2, pin_memory=True
        )
        optimizer = torch.optim.AdamW(attribute_model.parameters(), lr=1e-4, weight_decay=1e-4)
        scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=CONFIG['reid_epochs'])
        criterion = nn.CrossEntropyLoss(label_smoothing=0.05)
        best_score, start_epoch, patience_count = -1.0, 0, 0
        if ATTR_LAST.is_file():
            try:
                resume = torch.load(ATTR_LAST, map_location=DEVICE, weights_only=False)
                attribute_model.load_state_dict(resume['state_dict'])
                optimizer.load_state_dict(resume['optimizer'])
                if 'scheduler' in resume:
                    scheduler.load_state_dict(resume['scheduler'])
                start_epoch = int(resume.get('epoch', 0))
                best_score = float(resume.get('best_score', -1.0))
                status('ok', f'استئناف تدريب السمات من epoch {start_epoch}')
            except Exception as resume_exc:
                status('pause', f'تعذر استئناف checkpoint؛ سيبدأ epoch جديد: {resume_exc}')

        best_state = None
        last_metrics = None
        for epoch in range(start_epoch, CONFIG['reid_epochs']):
            attribute_model.train()
            train_loss = 0.0
            for images, identity, color, vehicle_type in train_loader:
                images = images.to(DEVICE, non_blocking=True)
                identity = identity.to(DEVICE); color = color.to(DEVICE); vehicle_type = vehicle_type.to(DEVICE)
                outputs = attribute_model(images)
                loss = (
                    0.25 * criterion(outputs['identity'], identity)
                    + criterion(outputs['color'], color)
                    + criterion(outputs['type'], vehicle_type)
                )
                optimizer.zero_grad(set_to_none=True)
                loss.backward()
                torch.nn.utils.clip_grad_norm_(attribute_model.parameters(), 5.0)
                optimizer.step()
                train_loss += float(loss.detach().cpu())
            scheduler.step()

            attribute_model.eval()
            color_true, color_pred, type_true, type_pred = [], [], [], []
            with torch.no_grad():
                for images, identity, color, vehicle_type in val_loader:
                    outputs = attribute_model(images.to(DEVICE, non_blocking=True))
                    color_true.extend(color.tolist())
                    color_pred.extend(outputs['color'].argmax(1).cpu().tolist())
                    type_true.extend(vehicle_type.tolist())
                    type_pred.extend(outputs['type'].argmax(1).cpu().tolist())

            color_acc = accuracy_score(color_true, color_pred)
            type_acc = accuracy_score(type_true, type_pred)
            score = (color_acc + type_acc) / 2
            last_metrics = {
                'color_accuracy': float(color_acc), 'type_accuracy': float(type_acc),
                'mean_attribute_accuracy': float(score),
                'color_balanced_accuracy': float(balanced_accuracy_score(color_true, color_pred)),
                'type_balanced_accuracy': float(balanced_accuracy_score(type_true, type_pred)),
                'color_macro_f1': float(f1_score(color_true, color_pred, average='macro', zero_division=0)),
                'type_macro_f1': float(f1_score(type_true, type_pred, average='macro', zero_division=0)),
                'color_confusion_matrix': confusion_matrix(color_true, color_pred).tolist(),
                'type_confusion_matrix': confusion_matrix(type_true, type_pred).tolist(),
                'n_validation': len(color_true), 'input_hw': CONFIG['reid_input_hw']
            }
            state = {
                'epoch': epoch + 1, 'state_dict': attribute_model.state_dict(),
                'optimizer': optimizer.state_dict(), 'scheduler': scheduler.state_dict(),
                'best_score': max(best_score, score), 'metrics': last_metrics,
                'cfg': CONFIG,
                'identity_classes': identity_encoder.classes_.tolist(),
                'color_classes': color_encoder.classes_.tolist(),
                'type_classes': type_encoder.classes_.tolist(),
                'source_reid_checkpoint': str(reid_checkpoint)
            }
            torch.save(state, ATTR_LAST)
            atomic_copy(ATTR_LAST, DRIVE_ROOT / f'reid_attr_{RUN_ID}_last.pt')
            if score > best_score:
                best_score = score
                best_state = copy.deepcopy(attribute_model.state_dict())
                torch.save(state, ATTR_BEST)
                atomic_copy(ATTR_BEST, DRIVE_ROOT / f'reid_attr_{RUN_ID}_best.pt')
                patience_count = 0
            else:
                patience_count += 1
            print(
                f"epoch={epoch+1}/{CONFIG['reid_epochs']} "
                f"loss={train_loss/max(1,len(train_loader)):.4f} "
                f"color_acc={color_acc:.4f} type_acc={type_acc:.4f}"
            )
            if patience_count >= 8:
                print('Early stopping: لم يتحسن متوسط دقة السمات.')
                break

        if ATTR_BEST.is_file():
            best_ck = torch.load(ATTR_BEST, map_location=DEVICE, weights_only=False)
            attribute_model.load_state_dict(best_ck['state_dict'])
            save_json(best_ck.get('metrics', last_metrics), OUTPUT_ROOT / 'attribute_metrics.json')
        status('ok', 'تم حفظ أفضل وآخر checkpoint للسمات مع مقاييس validation')
    except Exception as exc:
        status('error', f'فشل تدريب attributes: {type(exc).__name__}: {exc}')
else:
    status('pause', 'تدريب السمات يحتاج manifest صالحاً وcheckpoint ReID وبيانات validation وGPU')


epoch=1/30 loss=3.6219 color_acc=0.2000 type_acc=0.2000
epoch=2/30 loss=3.6219 color_acc=0.2000 type_acc=0.2000
epoch=3/30 loss=3.6219 color_acc=0.2000 type_acc=0.2000
epoch=4/30 loss=3.6219 color_acc=0.2000 type_acc=0.2000
epoch=5/30 loss=3.6218 color_acc=0.2000 type_acc=0.2000
epoch=6/30 loss=3.6218 color_acc=0.2000 type_acc=0.2000
epoch=7/30 loss=3.6218 color_acc=0.2000 type_acc=0.2000
epoch=8/30 loss=3.6218 color_acc=0.2000 type_acc=0.2000
epoch=9/30 loss=3.6218 color_acc=0.2000 type_acc=0.2000
Early stopping: لم يتحسن متوسط دقة السمات.
✅ تم حفظ أفضل وآخر checkpoint للسمات مع مقاييس validation


In [45]:
# ١٤. التسميات اليدوية للوحات اليمنية — تُحفظ داخل Drive مع provenance
import cv2, json
from pathlib import Path

PLATE_LABEL_ROOT = RUN_ROOT / 'plate_labels'
PLATE_LABEL_ROOT.mkdir(parents=True, exist_ok=True)
PLATE_JSONL = PLATE_LABEL_ROOT / 'manual_labels.jsonl'


def annotate_plate_colab(image_path):
    # شغّل هذه الدالة لاحقاً في Colab لكل صورة مصرح بها؛ لا تُستدعى تلقائياً.
    import matplotlib.pyplot as plt
    image_path = Path(image_path)
    image = plt.imread(image_path)
    figure, axis = plt.subplots(figsize=(12, 7))
    axis.imshow(image); axis.set_title('انقر أعلى يسار اللوحة ثم أسفل يمينها')
    points = plt.ginput(2, timeout=0)
    plt.close(figure)
    if len(points) != 2:
        raise ValueError('يجب تحديد نقطتين')
    text = input('النص الأرضي للوحة (العربي/الإنجليزي): ').strip()
    height, width = image.shape[:2]
    (x1, y1), (x2, y2) = points
    x1, x2 = sorted([max(0, min(width, x1)), max(0, min(width, x2))])
    y1, y2 = sorted([max(0, min(height, y1)), max(0, min(height, y2))])
    row = {'image': str(image_path), 'bbox_xyxy': [x1, y1, x2, y2], 'text': text, 'annotator': 'manual', 'created_at_utc': datetime.now(timezone.utc).isoformat()}
    with open(PLATE_JSONL, 'a', encoding='utf-8') as handle:
        handle.write(json.dumps(row, ensure_ascii=False) + '\n')
    status('ok', f'حُفظت تسمية لوحة: {image_path.name}')

status('ok', f'أداة التسميات اليدوية جاهزة في {PLATE_LABEL_ROOT}')
status('pause', 'لم تُنشأ labels تلقائياً؛ شغّل annotate_plate_colab على صور مصرح بها')

✅ أداة التسميات اليدوية جاهزة في /content/drive/MyDrive/YemenVision_MTMC/runs/completion_1_3_20261117T183727Z/plate_labels
⏸️ لم تُنشأ labels تلقائياً؛ شغّل annotate_plate_colab على صور مصرح بها


In [48]:
# ١٥. إعداد data.yaml للوحات وتدريب كاشف plate — labels حقيقية فقط
PLATE_YAML = None  # عيّن مسار data.yaml بعد تحويل التسميات إلى YOLO
PLATE_RUN = RUN_ROOT / 'plate_detector'
PLATE_RUN.mkdir(parents=True, exist_ok=True)

if PLATE_YAML and Path(PLATE_YAML).is_file() and torch.cuda.is_available():
    try:
        from ultralytics import YOLO
        plate_model = YOLO('yolo11n.pt')
        plate_model.train(data=str(PLATE_YAML), imgsz=CONFIG['plate_imgsz'], epochs=60, batch=-1, device=0, workers=2, amp=True, save_period=5, patience=15, project=str(RUN_ROOT), name='plate_detector', exist_ok=True, plots=True)
        plate_best = PLATE_RUN / 'weights' / 'best.pt'
        plate_last = PLATE_RUN / 'weights' / 'last.pt'
        if plate_best.is_file(): atomic_copy(plate_best, DRIVE_ROOT / f'plate_yolo_{RUN_ID}_best.pt')
        if plate_last.is_file(): atomic_copy(plate_last, DRIVE_ROOT / f'plate_yolo_{RUN_ID}_last.pt')
        status('ok', 'تم حفظ كاشف اللوحات بإصدار جديد')
    except Exception as exc:
        status('error', f'فشل تدريب كاشف اللوحات: {exc}')
else:
    status('pause', 'لا يوجد data.yaml للوحات؛ لا يُستخدم كاشف المركبات كبديل صامت')

⏸️ لا يوجد data.yaml للوحات؛ لا يُستخدم كاشف المركبات كبديل صامت


In [47]:
# ١٦. OCR للوحات: قراءة crop من التسميات اليدوية وتقييم CER وExact Plate Accuracy
import re, unicodedata, json
from pathlib import Path

DIGIT_TRANSLATION = str.maketrans(
    '٠١٢٣٤٥٦٧٨٩۰۱۲۳۴۵۶۷۸۹', '01234567890123456789'
)

def normalize_plate(text):
    normalized = unicodedata.normalize('NFKC', str(text)).translate(DIGIT_TRANSLATION).upper()
    return re.sub(r'[^A-Z0-9\u0600-\u06FF]', '', normalized)

def edit_distance(a, b):
    # Levenshtein distance بذاكرة O(min(len(a), len(b))).
    if len(a) < len(b):
        a, b = b, a
    previous = list(range(len(b) + 1))
    for i, ca in enumerate(a, 1):
        current = [i]
        for j, cb in enumerate(b, 1):
            current.append(min(
                current[-1] + 1, previous[j] + 1,
                previous[j - 1] + (ca != cb)
            ))
        previous = current
    return previous[-1]

def character_accuracy(ground_truth, prediction):
    truth, predicted = normalize_plate(ground_truth), normalize_plate(prediction)
    denom = max(len(truth), len(predicted), 1)
    return max(0.0, 1.0 - edit_distance(truth, predicted) / denom)

ocr_engine = None
OCR_LANGUAGE = 'ar'
try:
    from paddleocr import PaddleOCR
    try:
        ocr_engine = PaddleOCR(use_angle_cls=True, lang=OCR_LANGUAGE, show_log=False)
    except Exception:
        OCR_LANGUAGE = 'en'
        ocr_engine = PaddleOCR(use_angle_cls=True, lang=OCR_LANGUAGE, show_log=False)
    status('ok', f'PaddleOCR جاهز؛ language={OCR_LANGUAGE}')
except Exception as exc:
    status('pause', f'OCR غير جاهز؛ ثبّت paddleocr وpaddlepaddle في Colab: {exc}')

def _flatten_ocr_text(result):
    found = []
    def walk(node):
        if isinstance(node, (list, tuple)):
            if len(node) >= 2 and isinstance(node[1], (list, tuple)) and node[1] and isinstance(node[1][0], str):
                found.append(node[1][0])
            else:
                for child in node:
                    walk(child)
    walk(result)
    return ''.join(found)

def run_ocr_on_manual_labels(jsonl_path=PLATE_JSONL):
    if ocr_engine is None:
        raise RuntimeError('PaddleOCR غير جاهز.')
    import cv2
    rows = []
    jsonl_path = Path(jsonl_path)
    if not jsonl_path.is_file():
        raise FileNotFoundError(f'لا يوجد ملف التسميات: {jsonl_path}')
    with open(jsonl_path, encoding='utf-8') as handle:
        labels = [json.loads(line) for line in handle if line.strip()]
    for item in labels:
        image_path = Path(item['image'])
        if not image_path.is_file() or not item.get('text'):
            continue
        image = cv2.imread(str(image_path))
        if image is None:
            continue
        x1, y1, x2, y2 = [int(round(v)) for v in item['bbox_xyxy']]
        h, w = image.shape[:2]
        x1, x2 = max(0, min(w, x1)), max(0, min(w, x2))
        y1, y2 = max(0, min(h, y1)), max(0, min(h, y2))
        crop = image[y1:y2, x1:x2]
        if crop.size == 0:
            continue
        raw = ocr_engine.ocr(crop, cls=True)
        prediction = _flatten_ocr_text(raw)
        rows.append({
            'image': str(image_path), 'gt': item['text'],
            'pred': prediction, 'character_accuracy': character_accuracy(item['text'], prediction)
        })
    if not rows:
        raise ValueError('لم توجد قصاصات لوحات صالحة ذات ground truth.')
    plate_accuracy = sum(
        normalize_plate(row['gt']) == normalize_plate(row['pred']) for row in rows
    ) / len(rows)
    metrics = {
        'exact_plate_accuracy': plate_accuracy,
        'mean_character_accuracy': sum(r['character_accuracy'] for r in rows) / len(rows),
        'n_evaluated': len(rows), 'language': OCR_LANGUAGE,
        'source': str(jsonl_path), 'predictions': rows
    }
    save_json(metrics, OUTPUT_ROOT / 'ocr_metrics.json')
    return metrics

status('pause', 'شغّل run_ocr_on_manual_labels() بعد إضافة تسميات يدوية حقيقية؛ لا تُحسب دقة قبل ذلك')


⏸️ OCR غير جاهز؛ ثبّت paddleocr وpaddlepaddle في Colab: No module named 'paddleocr'
⏸️ شغّل run_ocr_on_manual_labels() بعد إضافة تسميات يدوية حقيقية؛ لا تُحسب دقة قبل ذلك


In [49]:
# ١٧. تمويه اللوحات قبل كتابة الفيديو — خصوصية افتراضية
import cv2


def blur_plate_boxes(frame, boxes, kernel_size=31):
    result = frame.copy()
    height, width = result.shape[:2]
    kernel = max(3, int(kernel_size) // 2 * 2 + 1)
    for x1, y1, x2, y2 in boxes:
        x1, y1 = max(0, int(x1)), max(0, int(y1))
        x2, y2 = min(width, int(x2)), min(height, int(y2))
        if x2 > x1 and y2 > y1:
            result[y1:y2, x1:x2] = cv2.GaussianBlur(result[y1:y2, x1:x2], (kernel, kernel), 0)
    return result

PRIVACY_BLUR_PLATES = True
status('ok', 'تم تجهيز blur اختياري؛ فعّله قبل حفظ أي فيديو أو صورة')

✅ تم تجهيز blur اختياري؛ فعّله قبل حفظ أي فيديو أو صورة


In [50]:
# ١٨. ملخص المراحل ١–٣ — لا تُسجل الدقة إلا بعد تقييم حقيقي
summary = {
    'run_id': RUN_ID,
    'legacy_notebook': 'YemenVision_MTMC_Colab_Fixed (1).ipynb',
    'input_models': model_inventory,
    'detector_baseline': str(OUTPUT_ROOT / 'detector_baseline_metrics.json')
        if (OUTPUT_ROOT / 'detector_baseline_metrics.json').is_file() else 'غير مقيّم',
    'detector_fine_tuned': str(OUTPUT_ROOT / 'detector_metrics.json')
        if (OUTPUT_ROOT / 'detector_metrics.json').is_file() else 'غير مقيّم',
    'vehicle_attributes': str(OUTPUT_ROOT / 'attribute_metrics.json')
        if (OUTPUT_ROOT / 'attribute_metrics.json').is_file() else 'غير مقيّم',
    'plate_detector': 'غير مقيّم — يلزم dataset لوحات حقيقي بصيغة YOLO',
    'plate_ocr': str(OUTPUT_ROOT / 'ocr_metrics.json')
        if (OUTPUT_ROOT / 'ocr_metrics.json').is_file() else 'غير مقيّم — يلزم تسميات يدوية وOCR',
    'mtmc': 'غير مقيّم — يلزم فيديوهات متزامنة وهويات مرجعية ومقياس تقييم',
    'privacy_blur_plates': CONFIG['privacy_blur_plates'],
    'notes': [
        'النماذج القديمة لم تُستبدل.',
        'Fine-tuning يبدأ من checkpoint detector الأصلي نفسه.',
        'تقسيم train/val/test يحافظ على تقسيم المصدر لتقليل تسرب البيانات.',
        'لا يمكن ضمان دقة 100%؛ تقاس التحسينات على validation/test مستقل.'
    ]
}
save_json(summary, OUTPUT_ROOT / 'stages_1_3_summary.json')
print(json.dumps(summary, ensure_ascii=False, indent=2))
status('ok', f'سجل الإصدار محفوظ في {OUTPUT_ROOT}')


{
  "run_id": "20261117T183727Z",
  "legacy_notebook": "YemenVision_MTMC_Colab_Fixed (1).ipynb",
  "input_models": {
    "detector_best": {
      "path": "/content/drive/MyDrive/YemenVision_MTMC/best.pt",
      "bytes": 19168794,
      "sha256": "6f5354be2e8dafe98b187e25a2fd582340c2a32fd7a8a51e550db6680624eb8d",
      "source": "legacy_notebook_drive_artifact"
    },
    "detector_last": {
      "path": "/content/drive/MyDrive/YemenVision_MTMC/detection_yolo11s_last.pt",
      "bytes": 94910883,
      "sha256": "9917cde5fcec13b18cbebeca60efcc4cf55e28d118647208bbf2cdb88332ac28",
      "source": "legacy_notebook_drive_artifact"
    },
    "reid_best": {
      "path": "/content/drive/MyDrive/YemenVision_MTMC/reid_bot_best.pt",
      "bytes": 99106867,
      "sha256": "41e35b774e96df0b58dd3063e6586ced1c1ead85bf32f14732974c3e6ed6202b",
      "source": "legacy_notebook_drive_artifact"
    },
    "reid_last": {
      "path": "/content/drive/MyDrive/YemenVision_MTMC/reid_bot_last.pt",
      "b